# Q1 -- Which markets are the most efficient, and why?

One table, 33 rows, four measures. Self-contained: re-reads the parquet files, so
notebooks 1 and 2 do not have to be run first.

**Why these four measures**

| measure | why |
| --- | --- |
| time-weighted relative spread | raw cents cannot rank anything -- the tick is 1c and the book sits at 1 tick in 87% of rows, so median spread is 0.01 almost everywhere. Dividing by mid separates them. |
| time-weighted cost to cross | half-spread + the `p(1-p)*0.07` taker fee. The fee peaks at p=0.50 and vanishes at the extremes, so it reorders the ranking. |
| median top-5 depth (contracts) | every contract settles at $0 or $1, so face value is identical across all 33 markets and quantity is already the notional-equivalent. Unlike stocks, no need to multiply by price. |
| total volume traded (contracts) | quotes are cheap to post; volume is the evidence that anyone acted on them. |

**Time-weighted** means each quote is weighted by how long it stood before the next
message replaced it. The feed only sends a message when something changes, so rows are
events, not clock ticks, and message counts differ 12x across markets. An unweighted
average would partly be measuring how chatty each market's feed is.

In [ ]:
import numpy as np
import pandas as pd

from utils import (time_weighted_relative_spread,
                   time_weighted_cost_to_cross,
                   median_total_quantity_top_5,
                   total_volume_traded)

df_trades = pd.read_parquet("trades_823753_pregame.parquet")
df_books  = pd.read_parquet("orderbook_data_823753_pregame.parquet")
for df in (df_trades, df_books):
    df["recv_ts_utc"] = pd.to_datetime(df["recv_ts_utc"], utc = True)

print("books ", df_books.shape)
print("trades", df_trades.shape)

In [ ]:
# Apply the four metrics to each of the 33 markets in turn.
# mid and n_trades are carried along as context -- they are not efficiency measures,
# but the table cannot be read without knowing where each market sits on the ladder.
rows = []
for mkt in sorted(df_books["native_id"].unique()):
    b_m = df_books[df_books["native_id"] == mkt]
    t_m = df_trades[df_trades["native_id"] == mkt]
    rows.append({
        "market":       mkt.replace("-26AUG051940PITMIL", ""),   # strip the game stamp
        "mid":          ((b_m["best_bid"] + b_m["best_ask"]) / 2).median(),
        "rel_spread_%": 100 * time_weighted_relative_spread(b_m),
        "cost_cross_%": 100 * time_weighted_cost_to_cross(b_m),
        "depth_top5":   median_total_quantity_top_5(b_m),
        "volume":       total_volume_traded(t_m),
        "n_trades":     len(t_m),
    })

q1 = pd.DataFrame(rows).set_index("market")
q1.sort_values("volume", ascending = False).round(2)

### Ranked by each measure separately -- they disagree, which is the interesting part

In [ ]:
for col, asc in [("rel_spread_%", True), ("cost_cross_%", True),
                 ("depth_top5", False), ("volume", False)]:
    top = q1.sort_values(col, ascending = asc).head(5)
    print(f"--- best 5 by {col} ---")
    print(top[[ "mid", col ]].round(2).to_string(), "\n")

## Answer

**The most efficient market is `KXMLBRFI`** -- will there be at least 1 run in the
1st inning.

| | RFI | next best |
| --- | --- | --- |
| volume | 272,584 contracts | 98,746 (TOTAL-8) -- **2.8x** |
| trades | 1,234 | 385 (TOTAL-8) -- **3.2x** |
| top-5 depth | 3,232,711 contracts | 444,485 (TOTAL-8) -- **7.3x** |
| relative spread | 2.47% | mid-pack |

Depth and volume are not close. RFI is quoted seven times deeper than any other market
and trades nearly three times as much as the next one.

**Why it is the most efficient.** It is the only one of the 33 that is a standalone
binary rather than one strike on a ladder -- there is a single number to have an opinion
about, not a distribution to price consistently. It also resolves first, in the top of
the 1st inning, so capital is committed for the shortest time and the payoff is nearest.
Both effects concentrate attention and capital in one contract, and that is what
efficiency is: many participants pricing the same simple question.

**Its relative spread is only mid-pack, and that is the honest caveat.** At 2.47% it is
beaten by TEAMTOTAL-PIT4 (2.36%) -- a market that traded 525 contracts against RFI's
272,584. Tight quotes are cheap to post. Volume and depth are what show that the quotes
were real.

### The ranking inverts on cost to cross, and the reason is the fee

The cheapest market to trade is **TOTAL-3 at 0.83% of mid**, with TOTAL-2 at 1.00% --
both trading near 0.96. That is not good market-making, it is the fee formula: at
p = 0.96, `p(1-p)*0.07` is 0.27 cents, while at p = 0.50 it is 1.75 cents, or 3.5x the
half-spread. Contracts priced near certainty are cheap to cross because the exchange
barely charges for them.

Neither traded much -- 38 and 6 trades. So "cheapest to cross" and "most efficient" are
different questions, and a ranking built on spread alone would have put the wrong
markets on top.

### Least efficient

The extreme TEAMTOTAL strikes. `TEAMTOTAL-MIL8` (mid 0.10) has a **19.5%** relative
spread and one trade; `PIT8` (mid 0.08) is 12.8% with one trade; `PIT6` and `PIT7` are
quoted continuously and **never trade at all**. Low-probability strikes on a single
team's run total attract no attention, so the quotes stay wide and nothing crosses.

### To do

- Speed of information incorporation: when a large trade hits one strike, how many
  seconds until neighbouring strikes in the same chain reprice? Tests whether prices
  actually absorb information rather than merely looking tidy. Best remaining Q1 idea.
- No-arbitrage violations net of fees: within-chain monotonicity, non-negative
  butterflies, RFI <= F5 <= TOTAL nesting.